# 06 Évaluation de la solution

La compétition [House Prices: Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) accepte encore des soumissions. Nous sommes donc dans le cas 1 des consignes : l'évaluation de la solution est celle de Kaggle, sur les 1 459 maisons de `test.csv`, dont les vrais prix ne sont pas publics.

Ce notebook suit les trois étapes demandées :
1. générer le fichier de prédiction (section 2);
2. soumettre la solution sur Kaggle (section 3);
3. fournir la capture d'écran de la soumission, le score obtenu et la position approximative au classement (section 4).

Le modèle évalué est celui retenu à l'[étape 5](05_solution.ipynb) : la moyenne simple de Lasso, ElasticNet, Kernel Ridge, Gradient Boosting et XGBoost, entraînés sur le prétraitement de l'[étape 4](04_pretraitement.ipynb).

## 1. Score attendu

Kaggle mesure la racine de la moyenne des carrés des écarts entre les logarithmes des prix prédits et observés :

$$
\text{RMSE log} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( \log \hat{p}_i - \log p_i \right)^2}
$$

où $n$ est le nombre de maisons, $p_i$ le prix observé de la maison $i$ et $\hat{p}_i$ son prix prédit. Plus le score est faible, meilleur il est.

Deux mesures de l'étape 5 donnent une idée du score à attendre :

| Mesure | Maisons évaluées | RMSE log |
|---|---|---|
| Validation croisée à cinq plis ([notebook 05](05_solution.ipynb), section 4) | 1 458 ventes de `train.csv`, sans les ventes 524 et 1299 | 0,1074 |
| Validation de l'[expérience 04](experiences/04_ensemble.ipynb), même ensemble de cinq modèles | 292 maisons jamais vues pendant les choix | 0,1233 |

Nous nous attendons à un score Kaggle plus proche de 0,12 que de 0,107, pour deux raisons :
- **la validation croisée exclut les deux ventes aberrantes**, alors que Kaggle compte toutes les maisons de `test.csv`. Celui-ci contient au moins une maison semblable (Id 2550 : 5 095 pi², qualité 10, vente partielle) : si elle a été vendue sous sa valeur, son erreur sera grande;
- **les scores de validation croisée ont servi à choisir le modèle** (section 4 du notebook 05) : ils sont légèrement optimistes. La validation de l'expérience 04, faite une seule fois après les choix, est une meilleure estimation, et elle était déjà de 0,123.

## 2. Fichier de prédiction

Le modèle est entraîné sur les 1 460 ventes de `train.csv`, sans les deux ventes aberrantes (524 et 1299), puis prédit le prix des 1 459 maisons de `test.csv`. Le fichier doit avoir le format de `sample_submission.csv` : une colonne `Id` et une colonne `SalePrice`, une ligne par maison de `test.csv`.

Le fichier soumis, `submissions/05_ensemble.csv`, a été produit par la section 6 du notebook 05. La cellule suivante le régénère avec le même code, vérifie son format, puis vérifie qu'il est identique au fichier soumis : la soumission est donc reproductible. Si le fichier n'existe pas encore, elle l'écrit.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.training import MEMBERS, fit_model, load_data, make_model

# =======================================================================
# Chargement des données
# =======================================================================
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
X, y, test = load_data(root / "data/train.csv", root / "data/test.csv")
sample = pd.read_csv(root / "data/sample_submission.csv")

# =======================================================================
# Entraînement final (cinq modèles) et prédiction de test.csv
# =======================================================================
final_model = fit_model(make_model(list(MEMBERS)), X, y)
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": final_model.predict(test)})

# =======================================================================
# Vérification du format
# =======================================================================
assert list(submission.columns) == list(sample.columns)
assert submission["Id"].equals(sample["Id"])
assert submission["SalePrice"].notna().all() and submission["SalePrice"].gt(0).all()

# =======================================================================
# Comparaison avec le fichier soumis (ou écriture s'il n'existe pas)
# =======================================================================
submission_path = root / "submissions" / "05_ensemble.csv"
if submission_path.exists():
    submitted = pd.read_csv(submission_path)
    assert submitted["Id"].equals(submission["Id"])
    gap = (submission["SalePrice"] / submitted["SalePrice"] - 1).abs().max()
    print(f"Écart maximal avec le fichier soumis : {gap:.2e} (relatif)")
    assert gap < 1e-6, "Le fichier régénéré diffère du fichier soumis."
else:
    submission_path.parent.mkdir(exist_ok=True)
    submission.to_csv(submission_path, index=False)

median = f"{submission['SalePrice'].median():,.0f}".replace(",", " ")
low, high = (f"{price:,.0f}".replace(",", " ")
             for price in submission["SalePrice"].agg(["min", "max"]))
print(f"{submission_path.relative_to(root)} : {len(submission)} prix, "
      f"médiane {median} $, de {low} $ à {high} $")

Écart maximal avec le fichier soumis : 0.00e+00 (relatif)
submissions\05_ensemble.csv : 1459 prix, médiane 157 893 $, de 46 231 $ à 913 862 $


## 3. Soumission sur Kaggle

Le fichier [05_ensemble.csv](../submissions/05_ensemble.csv) a été soumis le **27 novembre 2026**, depuis la page « Submit Prediction » de la compétition. C'est notre première soumission du modèle retenu.

Pour éviter d'ajuster le modèle au classement public, nous n'avons soumis que le modèle choisi à l'étape 5, sans le modifier après avoir vu son score. Soumettre plusieurs variantes et garder celle qui obtient le meilleur score public créerait un **biais de sélection** : le classement public ne porte que sur une partie de `test.csv`, et la variante retenue serait en partie celle qui a eu de la chance sur ces maisons. Son score public surestimerait alors sa performance réelle, comme un score de validation qui a servi à choisir le modèle ([Cawley et Talbot, 2010](https://jmlr.org/papers/v11/cawley10a.html)).

## 4. Résultat

<p align="left"><img src="../docs/figures/06_kaggle_soumission.png" alt="Classement public de la compétition House Prices après notre soumission : 1 179ᵉ rang, score 0,12246" width="900"></p>

| Élément | Valeur |
|---|---|
| Fichier soumis | [`submissions/05_ensemble.csv`](../submissions/05_ensemble.csv) |
| Date de soumission | 27 novembre 2026 |
| Score obtenu (RMSE log, classement public) | **0,12246** |
| Position au classement public | **1 179ᵉ** sur 4 968 participants (premiers 24 %) |
| Taille de la compétition | 4 968 participants, 19 955 soumissions |

**Position au classement.** Avec le 1 179ᵉ rang sur 4 968 participants, notre soumission se place dans les premiers 24 % du classement : elle fait mieux qu'environ trois participants sur quatre. Autour de notre position, les scores sont très serrés : les équipes classées 1 169ᵉ à 1 190ᵉ obtiennent 0,1224 et 0,1225. Un écart d'un millième fait donc changer le rang de plusieurs places.  Ce que représente un score compétitif pour cette compétition est discuté à l'[étape 7](07_analyse_critique.ipynb) (section 1.2).